# Local backups

`device.backup_to_storage(root=...)` writes the backups of a device to a local folder, with the same layout as in S3: `devices/<id>/data/*.parquet` (the data) and `devices/<id>/metadata.json` (the device from the Smart Citizen API). `device.load_from_storage(root=...)` reads the data back into a device: not its metadata. Later backups can have more channels: all of them are read when the types of their columns can be read together (as here).

It needs scdata 1.8.0 or later with the storage extra, but no S3 bucket or AWS credentials:

```
pip install 'scdata[storage]>=1.8.0'
```

In [1]:
import logging
import os
import tempfile
from os.path import join
from unittest import mock

import numpy as np
import pandas as pd
import scdata as sc
from scdata.tools.custom_logger import set_logger_level

# Only errors: 'WARNING', 'INFO' or 'DEBUG' show what scdata and smartcitizen-connector do
set_logger_level('ERROR')
logging.getLogger('smartcitizen_connector').setLevel('ERROR')

sc.__version__

Found Environment file at: /home/oscar/.cache/scdata/.env


'1.8.0'

In [2]:
# A new folder for each run of this notebook, removed at the end
folder = tempfile.TemporaryDirectory(prefix='local_backups_')
root = folder.name

def tree():
    for folder, _, files in sorted(os.walk(root)):
        for file in sorted(files):
            print(os.path.relpath(join(folder, file), root))

def parquet_files():
    return sorted(os.listdir(join(root, 'devices', str(device.id), 'data')))

def load(**kwargs):
    ''' The backups into stored, or an error '''
    if not stored.load_from_storage(root=root, **kwargs):
        raise RuntimeError(f'Cannot read the backups in {root}')

## Some data to back up

The device comes from the Smart Citizen API (its metadata goes to `metadata.json`). Its data here is three days of generated readings, one minute apart, so that the example does not depend on what the device sent.

In [3]:
device = sc.Device(params=sc.APIParams(id=18154))

index = pd.date_range('2026-01-01', periods=3 * 24 * 60, freq='1min', tz='UTC', name='TIME')
hours = np.arange(len(index)) / 60
data = pd.DataFrame({'TEMP': 15 + 5 * np.sin(2 * np.pi * hours / 24),
                     'HUM': 60 - 10 * np.sin(2 * np.pi * hours / 24)}, index=index)
days = [day for _, day in data.groupby(data.index.date)]
[len(day) for day in days]

[1440, 1440, 1440]

## Overwrite and append

`mode='overwrite'` replaces all the backups of the device, `mode='append'` (the default) adds one more file.

In [4]:
device.data = days[0]
device.backup_to_storage(mode='overwrite', root=root)
tree()

devices/18154/metadata.json
devices/18154/data/76c3aba6cd8e4d29ac239cfee97e49d8.parquet


In [5]:
device.data = days[1]
device.backup_to_storage(mode='append', root=root)
tree()

devices/18154/metadata.json
devices/18154/data/76c3aba6cd8e4d29ac239cfee97e49d8.parquet
devices/18154/data/acf25cb69aa14f42a0bc18d31f6d4729.parquet


In [6]:
stored = sc.Device(params=sc.APIParams(id=device.id))
load()
stored.data.index.min(), stored.data.index.max(), len(stored.data) == len(days[0]) + len(days[1])

(Timestamp('2026-01-01 00:00:00+0000', tz='UTC'),
 Timestamp('2026-01-02 23:59:00+0000', tz='UTC'),
 True)

## A channel added later

Each backup has the channels of that moment. The channels of all of them are read: a new one is empty before it was added.

In [7]:
device.data = days[2].assign(NEW=1.0)
device.backup_to_storage(root=root)

load()
stored.data['NEW'].isna().groupby(stored.data.index.date).mean()

2026-01-01    1.0
2026-01-02    1.0
2026-01-03    0.0
Name: NEW, dtype: float64

Only a period and some channels can be read (the filter goes to the parquet files). The dates keep their timezone:

In [8]:
load(min_date=days[1].index[0], channels=['TEMP', 'NEW'])
stored.data.index.min(), list(stored.data.columns)

(Timestamp('2026-01-02 00:00:00+0000', tz='UTC'), ['TEMP', 'NEW'])

## A failed backup keeps the previous one

The data and `metadata.json` are written aside first and then put in place. Here, two failures: writing the data (the disk is full), and replacing `metadata.json` once the data is in place (the data is put back). In both, the backups are as they were: the same data and the same files.

In [9]:
load()
before, files = stored.data.copy(), parquet_files()

def full(*args, **kwargs):
    raise OSError('No space left on device')

device.data = days[0]
for mode in ['overwrite', 'append']:
    with mock.patch.object(pd.DataFrame, 'to_parquet', full):
        try:
            device.backup_to_storage(mode=mode, root=root)
        except OSError as error:
            print(mode, 'failed:', error)

load()
stored.data.equals(before), parquet_files() == files

overwrite failed: No space left on device
append failed: No space left on device


(True, True)

In [10]:
real_replace = os.replace
def replace(source, destination):
    if destination.endswith('metadata.json'):
        raise OSError('Permission denied')
    real_replace(source, destination)

for mode in ['overwrite', 'append']:
    with mock.patch('os.replace', replace):
        try:
            device.backup_to_storage(mode=mode, root=root)
        except OSError as error:
            print(mode, 'failed:', error)

load()
stored.data.equals(before), parquet_files() == files

overwrite failed: Permission denied
append failed: Permission denied


(True, True)

## Only in S3

qc data and metrics are only backed up to S3: with a local `root`, it returns `False`.

In [11]:
device.backup_to_storage(root=root, add_qc_data=True)

[2026-11-18 20:27:25,057] - scdata - ERROR - qc data and metrics are only backed up to S3 (device.py:1293)


False

In [12]:
folder.cleanup()